In [7]:
import pandas as pd
import numpy as np

supplier_master = pd.DataFrame({
    "supplier_id": ["SUP_1", "SUP_2", "SUP_3", "SUP_4", "SUP_5"],
    "delay_probability": [0.05, 0.07, 0.10, 0.12, 0.15]
})

supplier_master

,supplier_id,delay_probability
0,SUP_1,0.05
1,SUP_2,0.07
2,SUP_3,0.10
3,SUP_4,0.12
4,SUP_5,0.15


In [8]:
sales = pd.read_csv("../data/chainos.db/sales.csv")

In [9]:
unique_items = sales["item_id"].unique()

np.random.seed(42)

sku_supplier_map = pd.DataFrame({
    "item_id": unique_items,
    "supplier_id": np.random.choice(
        supplier_master["supplier_id"],
        size=len(unique_items)
    )
})

sku_supplier_map.head(500)

,item_id,supplier_id
0,FOODS_3_090,SUP_4
1,FOODS_3_586,SUP_5
2,FOODS_3_252,SUP_3
3,FOODS_3_714,SUP_5
4,FOODS_3_120,SUP_5
5,FOODS_3_808,SUP_2
6,FOODS_3_587,SUP_3
7,FOODS_3_555,SUP_3
8,FOODS_3_080,SUP_3
9,FOODS_3_541,SUP_5


In [10]:
supplier_deliveries = sales.merge(
    sku_supplier_map,
    on="item_id",
    how="left"
)

In [13]:
np.random.seed(42)

base_lead_time = np.random.lognormal(
    mean=1.9,
    sigma=0.25,
    size=len(supplier_deliveries)
)

In [14]:
supplier_deliveries["lead_time_days"] = (
    np.round(base_lead_time)
    .clip(4, 12)
    .astype(int)
)

In [15]:
supplier_deliveries["lead_time_days"].describe()

count    13409.000000
mean         6.889776
std          1.724047
min          4.000000
25%          6.000000
50%          7.000000
75%          8.000000
max         12.000000
Name: lead_time_days, dtype: float64

In [16]:
supplier_deliveries = supplier_deliveries.merge(
    supplier_master,
    on="supplier_id",
    how="left"
)

In [17]:
supplier_deliveries["rand"] = np.random.rand(
    len(supplier_deliveries)
)

In [18]:
supplier_deliveries["delayed_flag"] = (
    supplier_deliveries["rand"]
    < supplier_deliveries["delay_probability"]
).astype(int)

In [19]:
supplier_deliveries["delay_days"] = np.where(
    supplier_deliveries["delayed_flag"] == 1,
    np.random.randint(
        7,
        15,
        size=len(supplier_deliveries)
    ),
    0
)

In [20]:
supplier_deliveries["actual_lead_time"] = (
    supplier_deliveries["lead_time_days"]
    + supplier_deliveries["delay_days"]
)

In [21]:
supplier_deliveries = supplier_deliveries[
    [
        "item_id",
        "store_id",
        "wm_yr_wk",
        "supplier_id",
        "lead_time_days",
        "delay_days",
        "actual_lead_time",
        "delayed_flag"
    ]
]

In [22]:
supplier_deliveries["delayed_flag"].mean()

np.float64(0.1013498396599299)

In [23]:
supplier_scorecard = (
    supplier_deliveries
    .groupby("supplier_id")
    .agg(
        avg_lead_time=("actual_lead_time","mean"),
        delay_rate=("delayed_flag","mean")
    )
    .reset_index()
)

supplier_scorecard

,supplier_id,avg_lead_time,delay_rate
0,SUP_1,7.444507,0.050141
1,SUP_2,7.544238,0.066171
2,SUP_3,7.919755,0.098968
3,SUP_4,8.138241,0.120820
4,SUP_5,8.459899,0.145592


In [26]:
supplier_deliveries.to_csv(
    "../data/chainos.db/supplier_deliveries.csv",
    index=False
)

supplier_scorecard.to_csv('../data/chainos.db/supplier_scorecard.csv',
    
    index=False
)